In [108]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [109]:
import nltk
from nltk.corpus import stopwords

In [110]:
nltk.download("stopwords")
nepali_stopwords = stopwords.words("nepali")

print(nepali_stopwords)

['छ', 'र', 'पनि', 'छन्', 'लागि', 'भएको', 'गरेको', 'भने', 'गर्न', 'गर्ने', 'हो', 'तथा', 'यो', 'रहेको', 'उनले', 'थियो', 'हुने', 'गरेका', 'थिए', 'गर्दै', 'तर', 'नै', 'को', 'मा', 'हुन्', 'भन्ने', 'हुन', 'गरी', 'त', 'हुन्छ', 'अब', 'के', 'रहेका', 'गरेर', 'छैन', 'दिए', 'भए', 'यस', 'ले', 'गर्नु', 'औं', 'सो', 'त्यो', 'कि', 'जुन', 'यी', 'का', 'गरि', 'ती', 'न', 'छु', 'छौं', 'लाई', 'नि', 'उप', 'अक्सर', 'आदि', 'कसरी', 'क्रमशः', 'चाले', 'अगाडी', 'अझै', 'अनुसार', 'अन्तर्गत', 'अन्य', 'अन्यत्र', 'अन्यथा', 'अरु', 'अरुलाई', 'अर्को', 'अर्थात', 'अर्थात्', 'अलग', 'आए', 'आजको', 'ओठ', 'आत्म', 'आफू', 'आफूलाई', 'आफ्नै', 'आफ्नो', 'आयो', 'उदाहरण', 'उनको', 'उहालाई', 'एउटै', 'एक', 'एकदम', 'कतै', 'कम से कम', 'कसै', 'कसैले', 'कहाँबाट', 'कहिलेकाहीं', 'का', 'किन', 'किनभने', 'कुनै', 'कुरा', 'कृपया', 'केही', 'कोही', 'गए', 'गरौं', 'गर्छ', 'गर्छु', 'गर्नुपर्छ', 'गयौ', 'गैर', 'चार', 'चाहनुहुन्छ', 'चाहन्छु', 'चाहिए', 'छू', 'जताततै', 'जब', 'जबकि', 'जसको', 'जसबाट', 'जसमा', 'जसलाई', 'जसले', 'जस्तै', 'जस्तो', 'जस्तोसुकै', 'जहाँ'

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/nameless/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [111]:
import numpy as np

In [112]:
with open("./dataset/news.txt", "r") as file:
    corpus = file.read()

In [ ]:
import re
# Preprocess the corpus
training_data = []

sentences = [re.sub(r"[^\u0900-\u0963\u0966-\u097F\s]", "", s).strip() for s in corpus.split('\n')]

sentences = sentences[:200]

In [114]:
vocab = list(set([w for s in sentences for w in s.split()]))
vocab.sort()
vocab_size = len(vocab)
print(vocab_size)
vocab[300]

6613


'आइतबार'

In [115]:
word_to_idx = {word: i for i, word in enumerate(vocab)}

In [116]:
def get_column_vector(word):
    idx = word_to_idx[word]
    vec = np.zeros((vocab_size, 1), dtype=np.int8)  # 1 byte per int instead of 8
    vec[idx, 0] = 1
    return vec

In [117]:
get_column_vector("सेनाले")

array([[0],
       [0],
       [0],
       ...,
       [0],
       [0],
       [0]], shape=(6613, 1), dtype=int8)

In [118]:
# generate training data
window_size = 2
training_data = []
for s in sentences:
    words = s.split()
    s_len = len(words)
    for i, w in enumerate(words):
        start = max(0, i - window_size)
        end = min(s_len, i + window_size + 1)
        for j in range(start,end):
            if i != j:
                data = (w, words[j])
                training_data.append(data)

len(training_data)

54496

In [119]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

In [120]:
class SkipGram():
    def __init__(self, vocab_size, embedding_size, learning_rate) -> None:
        self.input_size = vocab_size
        self.hidden_size = embedding_size
        self.output_size = vocab_size
        self.alpha = learning_rate

    def initializeNN(self, data):
        self.data = data
        self.W = np.random.uniform(-0.1, 0.1, (self.input_size, self.hidden_size))
        self.W1 = np.random.uniform(-0.1,0.1, (self.hidden_size, self.output_size))

    def forwardPass(self, X):
        self.h = np.dot(self.W.T, X)
        self.ouput_z = np.dot(self.W1.T, self.h)
        self.y = softmax(self.ouput_z)

    def backpropagate(self, x, t):
        e = self.y - t
        dLdW1 = np.dot(self.h, e.T)
        dLdW = np.dot(x, np.dot(self.W1, e).T)
        self.W1 = self.W1 - self.alpha*dLdW1
        self.W = self.W - self.alpha*dLdW

    def train(self, epochs):
        for i in range(0, epochs):
            self.loss = 0
            for j in range(len(self.data)):
                X = get_column_vector(self.data[j][0])
                y = get_column_vector(self.data[j][1])
                self.forwardPass(X)
                self.backpropagate(X,y)

                self.loss += -np.sum(y * np.log(self.y + 1e-9))

            avg_loss = self.loss / len(self.data)
            print(f"Epoch {i + 1}, Loss: {avg_loss:.4f}")
            self.alpha *= 1 / (1 + self.alpha * i)

    def get_embedding_matrix(self):
        return self.W

In [ ]:
skip_gram = SkipGram(vocab_size, 50, 0.01)
skip_gram.initializeNN(training_data)
skip_gram.train(10)

KeyboardInterrupt: 